In [ ]:
# Cell 1 — Environment check
# CM3015 Breast Cancer Detection
# Notebook 01: Dataset Setup and EDA

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
print(f"TensorFlow: {tf.__version__}")
print(f"GPUs: {tf.config.list_physical_devices('GPU')}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")

In [ ]:
# Cell 2 — Confirm what is attached
print("=== /kaggle/input contents ===")
for item in sorted(os.listdir('/kaggle/input')):
    print(f"\n[{item}]")
    subpath = f'/kaggle/input/{item}'
    try:
        contents = sorted(os.listdir(subpath))
        for f in contents[:8]:  # show first 8 items only
            print(f"    {f}")
        if len(contents) > 8:
            print(f"    ... and {len(contents)-8} more")
    except:
        print("    (could not read)")

In [ ]:
# Cell 3 — Deep folder inspection
import os

def show_tree(path, depth=0, max_depth=4, max_files=5):
    if depth > max_depth:
        return
    try:
        items = sorted(os.listdir(path))
        files = [i for i in items if os.path.isfile(f'{path}/{i}')]
        dirs  = [i for i in items if os.path.isdir(f'{path}/{i}')]
        
        for d in dirs:
            print('    ' * depth + f'[{d}/]')
            show_tree(f'{path}/{d}', depth+1, max_depth, max_files)
        
        for f in files[:max_files]:
            size = os.path.getsize(f'{path}/{f}')
            print('    ' * depth + f'{f} ({size/1024:.1f} KB)')
        
        if len(files) > max_files:
            print('    ' * depth + f'... and {len(files)-max_files} more files')
    except Exception as e:
        print('    ' * depth + f'ERROR: {e}')

print("=== FULL INPUT TREE ===")
show_tree('/kaggle/input', max_depth=4)

In [ ]:
# Cell 4 — Define all paths
# Based on confirmed folder structure from Cell 3

# Base paths
AWSAF_BASE  = '/kaggle/input/datasets/awsaf49/cbis-ddsm-breast-cancer-image-dataset'
LABELS_BASE = '/kaggle/input/datasets/mfjmrizvi/cbis-ddsm-tcia-labels'
OUTPUT      = '/kaggle/working'

# CSV paths — using YOUR official TCIA versions
MASS_TRAIN_CSV = f'{LABELS_BASE}/mass_case_description_train_set.csv'
MASS_TEST_CSV  = f'{LABELS_BASE}/mass_case_description_test_set.csv'
CALC_TRAIN_CSV = f'{LABELS_BASE}/calc_case_description_train_set.csv'
CALC_TEST_CSV  = f'{LABELS_BASE}/calc_case_description_test_set.csv'

# Awsaf CSV paths (we use dicom_info.csv from here only)
DICOM_INFO_CSV = f'{AWSAF_BASE}/csv/dicom_info.csv'
META_CSV       = f'{AWSAF_BASE}/csv/meta.csv'

# Image root
JPEG_ROOT = f'{AWSAF_BASE}/jpeg'

# Verify all paths exist
paths = {
    'MASS_TRAIN_CSV': MASS_TRAIN_CSV,
    'MASS_TEST_CSV':  MASS_TEST_CSV,
    'CALC_TRAIN_CSV': CALC_TRAIN_CSV,
    'CALC_TEST_CSV':  CALC_TEST_CSV,
    'DICOM_INFO_CSV': DICOM_INFO_CSV,
    'META_CSV':       META_CSV,
    'JPEG_ROOT':      JPEG_ROOT,
}

print("=== PATH VERIFICATION ===")
all_ok = True
for name, path in paths.items():
    exists = os.path.exists(path)
    status = "✓" if exists else "✗ MISSING"
    print(f"  {status}  {name}: {path}")
    if not exists:
        all_ok = False

print(f"\nAll paths OK: {all_ok}")

In [ ]:
# Cell 5 — Load all CSV files

mass_train = pd.read_csv(MASS_TRAIN_CSV)
mass_test  = pd.read_csv(MASS_TEST_CSV)
calc_train = pd.read_csv(CALC_TRAIN_CSV)
calc_test  = pd.read_csv(CALC_TEST_CSV)
dicom_info = pd.read_csv(DICOM_INFO_CSV)
meta       = pd.read_csv(META_CSV)

print("=== SHAPES ===")
print(f"mass_train:  {mass_train.shape}")
print(f"mass_test:   {mass_test.shape}")
print(f"calc_train:  {calc_train.shape}")
print(f"calc_test:   {calc_test.shape}")
print(f"dicom_info:  {dicom_info.shape}")
print(f"meta:        {meta.shape}")

In [ ]:
# Cell 6 — Print exact column names
# Do not skip this — every downstream cell depends on these

print("=== MASS TRAIN COLUMNS ===")
for i, col in enumerate(mass_train.columns):
    print(f"  [{i:2d}] '{col}'")

print("\n=== CALC TRAIN COLUMNS ===")
for i, col in enumerate(calc_train.columns):
    print(f"  [{i:2d}] '{col}'")

print("\n=== DICOM INFO COLUMNS ===")
for i, col in enumerate(dicom_info.columns):
    print(f"  [{i:2d}] '{col}'")

print("\n=== META COLUMNS ===")
for i, col in enumerate(meta.columns):
    print(f"  [{i:2d}] '{col}'")

In [ ]:
# Cell 7 — Exact pathology string values

print("=== MASS TRAIN PATHOLOGY ===")
print(mass_train['pathology'].value_counts())
print("Unique:", mass_train['pathology'].unique())

print("\n=== MASS TEST PATHOLOGY ===")
print(mass_test['pathology'].value_counts())
print("Unique:", mass_test['pathology'].unique())

print("\n=== CALC TRAIN PATHOLOGY ===")
print(calc_train['pathology'].value_counts())
print("Unique:", calc_train['pathology'].unique())

print("\n=== CALC TEST PATHOLOGY ===")
print(calc_test['pathology'].value_counts())
print("Unique:", calc_test['pathology'].unique())

In [ ]:
# Cell 8 — Missing value audit

print("=== MISSING VALUES — MASS TRAIN ===")
m_missing = mass_train.isnull().sum()
print(m_missing[m_missing > 0])
print(f"Total missing cells: {m_missing.sum()}")

print("\n=== MISSING VALUES — CALC TRAIN ===")
c_missing = calc_train.isnull().sum()
print(c_missing[c_missing > 0])
print(f"Total missing cells: {c_missing.sum()}")

In [ ]:
# Cell 9 — Inspect dicom_info to understand image linking (fixed)

print("=== DICOM INFO FIRST 3 ROWS ===")
display(dicom_info.head(3))

print("\n=== SAMPLE SeriesInstanceUID values ===")
if 'SeriesInstanceUID' in dicom_info.columns:
    print(dicom_info['SeriesInstanceUID'].iloc[:3].tolist())

print("\n=== SAMPLE image file path from dicom_info ===")
for col in dicom_info.columns:
    non_null = dicom_info[col].dropna()
    if len(non_null) == 0:
        continue  # skip entirely null columns
    sample = str(non_null.iloc[0])
    if any(x in sample for x in
           ['1.3.6', 'jpeg', 'jpg', '/', 'CBIS', 'Mass', 'Calc']):
        print(f"  Column '{col}': {sample[:100]}")

In [ ]:
# Cell 10 — Load and display one image
# Confirms images are readable before we build any pipeline

import matplotlib.image as mpimg

# Get first folder in jpeg directory
jpeg_folders = sorted(os.listdir(JPEG_ROOT))
print(f"Total image folders: {len(jpeg_folders)}")
print(f"First folder name: {jpeg_folders[0]}")

# Go into first folder and find image
first_folder = f'{JPEG_ROOT}/{jpeg_folders[0]}'
subfolders = os.listdir(first_folder)
print(f"Contents of first folder: {subfolders}")

# Keep going deeper until we find a jpg file
def find_first_image(base_path, extensions=('.jpg', '.jpeg')):
    for root, dirs, files in os.walk(base_path):
        for f in files:
            if f.lower().endswith(extensions):
                return os.path.join(root, f)
    return None

img_path = find_first_image(JPEG_ROOT)
print(f"\nFirst image found: {img_path}")

if img_path:
    img = mpimg.imread(img_path)
    print(f"Image shape: {img.shape}")
    print(f"Image dtype: {img.dtype}")
    print(f"Pixel range: {img.min()} to {img.max()}")
    
    plt.figure(figsize=(6, 8))
    plt.imshow(img, cmap='gray')
    plt.title('First CBIS-DDSM image')
    plt.axis('off')
    plt.savefig(f'{OUTPUT}/first_image.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No image found — check folder structure")

In [ ]:
# Cell 11 — Class distribution plot

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('CBIS-DDSM Pathology Distribution', fontsize=16, fontweight='bold')

datasets = [
    (mass_train, 'Mass Train', axes[0,0]),
    (mass_test,  'Mass Test',  axes[0,1]),
    (calc_train, 'Calc Train', axes[1,0]),
    (calc_test,  'Calc Test',  axes[1,1]),
]

colors = ['#e74c3c', '#2ecc71', '#3498db']

for df, title, ax in datasets:
    counts = df['pathology'].value_counts()
    bars = ax.bar(counts.index, counts.values, color=colors[:len(counts)])
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Pathology')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=30)
    # Add count labels on bars
    for bar, count in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, 
                bar.get_height() + 5,
                str(count), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig(f'{OUTPUT}/pathology_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved")

In [ ]:
# Cell 12 — Binary label mapping
# BENIGN_WITHOUT_CALLBACK → merged with BENIGN
# Final: 0 = benign, 1 = malignant

def binarise_pathology(df):
    df = df.copy()
    df['label'] = df['pathology'].map({
        'MALIGNANT':             1,
        'BENIGN':                0,
        'BENIGN_WITHOUT_CALLBACK': 0
    })
    return df

mass_train = binarise_pathology(mass_train)
mass_test  = binarise_pathology(mass_test)
calc_train = binarise_pathology(calc_train)
calc_test  = binarise_pathology(calc_test)

print("=== BINARY LABEL DISTRIBUTION ===")
print("\nMass Train:")
print(mass_train['label'].value_counts())
print(f"Imbalance ratio: {mass_train['label'].value_counts()[0] / mass_train['label'].value_counts()[1]:.2f}:1 (benign:malignant)")

print("\nMass Test:")
print(mass_test['label'].value_counts())

print("\nCalc Train:")
print(calc_train['label'].value_counts())

print("\nCalc Test:")
print(calc_test['label'].value_counts())

# Check for any unmapped values
print("\nNull labels in mass_train:", mass_train['label'].isnull().sum())
print("Null labels in calc_train:", calc_train['label'].isnull().sum())

In [ ]:
# Cell 13 — Understand dicom_info SeriesDescription types

full_mammo = dicom_info[
    dicom_info['SeriesDescription'] == 'full mammogram images'
].copy()

print(f"Full mammogram entries: {len(full_mammo)}")
print(f"\nAll SeriesDescription values:")
print(dicom_info['SeriesDescription'].value_counts())

In [ ]:
# Cell 14 — Define extraction functions and build path map
# extract_series_uid extracts the SECOND UID from CSV paths
# CSV path format:
# Mass-Training_P_XXXXX/1.3.6.<StudyUID>/1.3.6.<SeriesUID>/000000.dcm
# We want the SeriesUID (second 1.3.6 segment)

def extract_series_uid(path_str):
    """Extract second SeriesInstanceUID from CBIS-DDSM CSV path strings."""
    if pd.isna(path_str):
        return None
    parts = str(path_str).replace('\\', '/').split('/')
    uids = [p for p in parts if p.startswith('1.3.6')]
    if len(uids) >= 2:
        return uids[1]  # second UID = SeriesUID
    elif len(uids) == 1:
        return uids[0]
    return None

def build_path_map(dicom_info_df, jpeg_root):
    """Build dict: SeriesUID -> full Kaggle JPEG path."""
    path_map = {}
    full_mammo = dicom_info_df[
        dicom_info_df['SeriesDescription'] == 'full mammogram images'
    ]
    for _, row in full_mammo.iterrows():
        img_path = str(row['image_path'])
        uid = extract_series_uid(img_path)
        if uid:
            filename = img_path.replace('\\', '/').split('/')[-1]
            path_map[uid] = f"{jpeg_root}/{uid}/{filename}"
    return path_map

# Build the map
path_map = build_path_map(dicom_info, JPEG_ROOT)

# Verify
print(f"Total paths in map: {len(path_map)}")

# Quick match check against mass_train
matches = sum(
    1 for val in mass_train['image file path']
    if extract_series_uid(val) in path_map
)
print(f"Matches in mass_train: {matches}/{len(mass_train)}")

# Sample entry
sample_uid = list(path_map.keys())[0]
print(f"\nSample path map entry:")
print(f"  UID:  {sample_uid}")
print(f"  Path: {path_map[sample_uid]}")
print(f"  File exists: {os.path.exists(path_map[sample_uid])}")

In [ ]:
# Cell 15 — Link images to CSV rows

def link_images_to_csv(df, path_map):
    """Add full_image_path column to dataframe using path_map."""
    df = df.copy()
    linked = []
    for _, row in df.iterrows():
        uid = extract_series_uid(str(row['image file path']))
        linked.append(path_map.get(uid, None))
    df['full_image_path'] = linked
    return df

mass_train_linked = link_images_to_csv(mass_train, path_map)
mass_test_linked  = link_images_to_csv(mass_test,  path_map)

train_linked = mass_train_linked['full_image_path'].notna().sum()
test_linked  = mass_test_linked['full_image_path'].notna().sum()

print(f"Mass train: {train_linked}/{len(mass_train_linked)} rows linked")
print(f"Mass test:  {test_linked}/{len(mass_test_linked)} rows linked")

linked_rows = mass_train_linked[mass_train_linked['full_image_path'].notna()]
if len(linked_rows) > 0:
    sample = linked_rows.iloc[0]
    print(f"\nSample linked row:")
    print(f"  patient_id:  {sample['patient_id']}")
    print(f"  pathology:   {sample['pathology']}")
    print(f"  label:       {sample['label']}")
    print(f"  image path:  {sample['full_image_path']}")
    print(f"  file exists: {os.path.exists(sample['full_image_path'])}")
else:
    print("ERROR: No linked rows found")

In [ ]:
# Cell 16 — Save linked dataframes for next notebook

mass_train_linked.to_csv(f'{OUTPUT}/mass_train_linked.csv', index=False)
mass_test_linked.to_csv(f'{OUTPUT}/mass_test_linked.csv',  index=False)

print("Saved:")
print(f"  {OUTPUT}/mass_train_linked.csv")
print(f"  {OUTPUT}/mass_test_linked.csv")
print(f"\nFinal mass_train_linked shape: {mass_train_linked.shape}")
print(f"Final mass_test_linked shape:  {mass_test_linked.shape}")
print(f"\nColumns: {mass_train_linked.columns.tolist()}")

In [ ]:
# Cell 17 — Final EDA summary save

import json

summary = {
    'dataset': 'CBIS-DDSM via Kaggle (awsaf49)',
    'labels_source': 'Official TCIA download',
    'scope': 'Mass cases only (prototype phase)',
    'label_mapping': {
        'MALIGNANT': 1,
        'BENIGN': 0,
        'BENIGN_WITHOUT_CALLBACK': 0
    },
    'mass_train': {
        'total_rows': len(mass_train_linked),
        'linked_to_image': int(train_linked),
        'label_0_benign': int(mass_train_linked['label'].value_counts()[0]),
        'label_1_malignant': int(mass_train_linked['label'].value_counts()[1]),
    },
    'mass_test': {
        'total_rows': len(mass_test_linked),
        'linked_to_image': int(test_linked),
        'label_0_benign': int(mass_test_linked['label'].value_counts()[0]),
        'label_1_malignant': int(mass_test_linked['label'].value_counts()[1]),
    },
    'image_properties': {
        'format': 'JPEG',
        'dtype': 'uint8',
        'pixel_range': '0-255',
        'sample_shape': '5696x4008 (varies per image)',
        'notes': 'Full mammogram images, high resolution'
    },
    'missing_values': {
        'mass_shape': 4,
        'mass_margins': 43,
        'impact': 'Low — descriptive columns only, not used in pipeline'
    }
}

with open(f'{OUTPUT}/eda_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("EDA complete. Summary saved.")
print(json.dumps(summary, indent=2))